# Lab 3: Image Segmentation & Deep Learning
**Tugas Lab - Computer Vision**

---

## 📚 Daftar Isi
1. **Setup & Import Libraries**
2. **BAGIAN 1 [60 pts]**: Morphology and Image Segmentation
   - 1(a): Morphological Preprocessing [10 pts]
   - 1(b): Canny Edge Detection [15 pts]
   - 1(c): Otsu Thresholding [15 pts]
   - 1(d): SLIC Superpixel Segmentation [15 pts]
   - 1(e): Analisis Perbandingan [5 pts]
3. **BAGIAN 2 [40 pts]**: Deep Neural Network Classification
   - 2(a): CNN Baseline [10 pts]
   - 2(b): Transfer Learning [20 pts]
   - 2(c): Analisis & Kesimpulan [10 pts]

---

## 1. Setup & Import Libraries

**📝 Panduan:**
- Untuk **Bagian 1** (Image Segmentation): Anda memerlukan `OpenCV`, `NumPy`, `Matplotlib`, dan `scikit-image`
- Untuk **Bagian 2** (Deep Learning): Anda bisa pilih **PyTorch** atau **TensorFlow/Keras**
- Di Kaggle, semua library ini sudah tersedia!

In [ ]:
# Import libraries untuk Image Processing
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import segmentation, color
from skimage.segmentation import slic, mark_boundaries
from skimage.util import img_as_float

# Import libraries untuk Deep Learning (pilih salah satu atau keduanya)
# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# TensorFlow/Keras (alternatif)
# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

print("✅ All libraries imported successfully!")
print(f"OpenCV version: {cv2.__version__}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Fungsi helper untuk visualisasi
def show_images(images, titles, figsize=(15, 5), cmap='gray'):
    """
    Menampilkan beberapa gambar dalam satu figure
    """
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    
    for ax, img, title in zip(axes, images, titles):
        if len(img.shape) == 3:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Helper functions defined!")

---

# BAGIAN 1: Morphology and Image Segmentation [60 poin]

## 🔬 Load Citra Sel Darah Merah

**📝 Instruksi:**
1. Upload file `blood-cells.jpg` ke Kaggle
2. Load citra menggunakan OpenCV
3. Convert ke grayscale untuk pemrosesan

In [ ]:
# Load citra
# CATATAN: Sesuaikan path dengan lokasi file Anda di Kaggle
# Contoh: '/kaggle/input/your-dataset-name/blood-cells.jpg'

img_path = '/kaggle/input/YOUR_DATASET_NAME/blood-cells.jpg'  # TODO: Ganti dengan path yang benar
img_original = cv2.imread(img_path)
img_gray = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

print(f"Image shape: {img_original.shape}")
print(f"Grayscale shape: {img_gray.shape}")

# Visualisasi
show_images([img_original, img_gray], 
            ['Original Image', 'Grayscale Image'],
            figsize=(12, 5))

---

## 1(a) Morphological Preprocessing [10 poin]

### 🎯 Tujuan:
Menghilangkan noise kecil dan memperjelas struktur sel darah menggunakan operasi morfologi

### 📚 Konsep Teori:

**Dari slide/referensi Anda:**

1. **Gaussian Blur** → Mengurangi noise dengan smoothing
2. **Opening** = Erosion → Dilation
   - Menghilangkan noise kecil
   - Formula: `f ∘ b = (f ⊖ b) ⊕ b`
3. **Closing** = Dilation → Erosion
   - Menutup lubang kecil di dalam objek
   - Formula: `f * b = (f ⊕ b) ⊖ b`

### 💡 Hint dari Soal:
> "Gunakan kombinasi filter Gaussian dan operasi morfologi dengan kernel berukuran kecil."

### 🔧 Contoh Sintaks (dari Tutorial):

In [ ]:
# ===== CONTOH KODE REFERENSI =====
# (Jangan langsung copy-paste! Pahami dulu konsepnya)

# 1. Gaussian Blur untuk mengurangi noise
# Sintaks: cv2.GaussianBlur(image, (kernel_size, kernel_size), sigmaX)
# Parameter:
#   - kernel_size: harus ganjil (3, 5, 7, 9, ...)
#   - sigmaX: standar deviasi untuk Gaussian (bisa 0 untuk auto)

# Contoh:
# blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)

# 2. Operasi Morfologi
# Langkah 1: Buat kernel (structuring element)
# Sintaks: cv2.getStructuringElement(shape, (width, height))
# Shape bisa: cv2.MORPH_RECT, cv2.MORPH_ELLIPSE, cv2.MORPH_CROSS

# Contoh:
# kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

# Langkah 2: Terapkan Opening atau Closing
# Sintaks Opening: cv2.morphologyEx(image, cv2.MORPH_OPEN, kernel)
# Sintaks Closing: cv2.morphologyEx(image, cv2.MORPH_CLOSE, kernel)

# Contoh:
# opened = cv2.morphologyEx(blurred, cv2.MORPH_OPEN, kernel)
# closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)

print("📖 Silakan implementasikan preprocessing Anda di bawah ini!")

In [ ]:
# TODO: Implementasi Morphological Preprocessing Anda

# Step 1: Gaussian Blur
# ...

# Step 2: Morphological Operations (Opening/Closing)
# ...

# Step 3: Visualisasi hasil sebelum dan sesudah
# Gunakan fungsi show_images() untuk membandingkan

# Step 4: Jelaskan efeknya dalam markdown cell berikutnya

**📝 Jelaskan hasil preprocessing Anda:**
- Operasi morfologi apa yang Anda gunakan?
- Apa efeknya terhadap kualitas citra?
- Apakah noise berkurang? Apakah struktur sel lebih jelas?

*(Tulis penjelasan Anda di sini)*

---

## 1(b) Canny Edge Detection [15 poin]

### 🎯 Tujuan:
Mendeteksi batas setiap sel darah menggunakan algoritma Canny

### 📚 Konsep Teori:

**Canny Edge Detection** adalah algoritma multi-tahap:
1. **Gaussian smoothing** → Mengurangi noise
2. **Gradient calculation** (Sobel) → Mencari magnitude & direction
3. **Non-maximum suppression** → Menipis tepi
4. **Double thresholding** → Menentukan tepi potensial
5. **Hysteresis** → Menghubungkan tepi dengan konektivitas

### 🔧 Sintaks Canny:

In [ ]:
# ===== CONTOH KODE REFERENSI =====

# Sintaks Canny Edge Detection:
# cv2.Canny(image, threshold1, threshold2)
#
# Parameter:
#   - image: Input image (grayscale)
#   - threshold1: Lower threshold untuk hysteresis
#   - threshold2: Upper threshold untuk hysteresis
#
# Rekomendasi: threshold2 = 2-3x threshold1

# Contoh:
# edges = cv2.Canny(preprocessed_image, threshold1=50, threshold2=150)

# 💡 Tips: 
# - Threshold terlalu rendah → banyak noise
# - Threshold terlalu tinggi → tepi hilang
# - Coba berbagai nilai untuk hasil terbaik!

print("📖 Implementasikan Canny edge detection Anda!")

In [ ]:
# TODO: Implementasi Canny Edge Detection

# Step 1: Terapkan Canny pada hasil preprocessing (1a)
# ...

# Step 2: Coba variasi threshold untuk hasil optimal
# ...

# Step 3: Visualisasi hasil
# ...

# Step 4: Analisis - apakah batas sel sudah terdeteksi dengan baik?

**📝 Analisis Hasil Canny:**
- Apakah batas tiap sel sudah terdeteksi dengan baik?
- Apakah ada tepi yang terputus?
- Nilai threshold berapa yang memberikan hasil terbaik?

*(Tulis analisis Anda di sini)*

---

## 1(c) Otsu Thresholding [15 poin]

### 🎯 Tujuan:
Mensegmentasi citra menggunakan metode thresholding otomatis Otsu

### 📚 Konsep Teori:

**Otsu's Method** mencari threshold optimal secara otomatis dengan:
- Memaksimalkan variance antar kelas (background & foreground)
- Meminimalkan variance dalam kelas

### 🔧 Sintaks Otsu:

In [ ]:
# ===== CONTOH KODE REFERENSI =====

# Sintaks Otsu Thresholding:
# ret, thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
#
# Parameter:
#   - image: Input grayscale image
#   - 0: Threshold value (diabaikan karena Otsu auto)
#   - 255: Maximum value
#   - cv2.THRESH_BINARY + cv2.THRESH_OTSU: Tipe thresholding
#
# Return:
#   - ret: Threshold value yang ditemukan Otsu
#   - thresh: Binary image hasil thresholding

# Contoh:
# ret, otsu_result = cv2.threshold(preprocessed_image, 0, 255, 
#                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
# print(f"Otsu threshold value: {ret}")

print("📖 Implementasikan Otsu thresholding!")

In [ ]:
# TODO: Implementasi Otsu Thresholding

# Step 1: Terapkan Otsu pada hasil preprocessing (1a)
# ...

# Step 2: Visualisasi hasil
# ...

# Step 3: Bandingkan dengan hasil Canny
# ...

**📝 Perbandingan Otsu vs Canny:**
- Pada kondisi seperti apa Otsu lebih unggul?
- Pada kondisi seperti apa Canny lebih unggul?
- Apa kelebihan dan kekurangan masing-masing metode?

*(Tulis perbandingan Anda di sini)*

---

## 1(d) SLIC Superpixel Segmentation [15 poin]

### 🎯 Tujuan:
Mensegmentasi citra menjadi superpixels menggunakan SLIC algorithm

### 📚 Konsep Teori:

**SLIC (Simple Linear Iterative Clustering)**:
- Mengelompokkan pixel berdasarkan color similarity dan spatial proximity
- Menghasilkan superpixels yang homogen

### 🔧 Sintaks SLIC:

In [ ]:
# ===== CONTOH KODE REFERENSI =====

# Sintaks SLIC dari scikit-image:
# segments = slic(image, n_segments=n, compactness=c, sigma=s)
#
# Parameter:
#   - image: Input image (bisa color atau grayscale)
#   - n_segments: Jumlah superpixels yang diinginkan (coba: 50, 100, 200)
#   - compactness: Trade-off antara color dan space (coba: 1, 10, 50)
#   - sigma: Gaussian smoothing sebelum segmentasi

# Contoh:
# segments = slic(img_as_float(preprocessed_image), 
#                 n_segments=100, compactness=10, sigma=1)

# Visualisasi boundaries:
# boundaries = mark_boundaries(original_image, segments)
# plt.imshow(boundaries)

# Superpixel mean image:
# mean_image = color.label2rgb(segments, original_image, kind='avg')
# plt.imshow(mean_image)

print("📖 Implementasikan SLIC superpixel segmentation!")

In [ ]:
# TODO: Implementasi SLIC Superpixel Segmentation

# Step 1: Terapkan SLIC dengan variasi parameter
# Coba beberapa nilai n_segments dan compactness
# ...

# Step 2: Visualisasi batas superpixel di atas citra asli
# ...

# Step 3: Buat superpixel mean image
# ...

# Step 4: Bandingkan hasil dari variasi parameter

**📝 Analisis SLIC:**
- Parameter mana yang memberikan hasil terbaik?
- Bagaimana pengaruh n_segments terhadap hasil?
- Bagaimana pengaruh compactness terhadap bentuk superpixels?

*(Tulis analisis Anda di sini)*

---

## 1(e) Analisis Perbandingan Hasil Segmentasi [5 poin]

### 🎯 Pertanyaan yang harus dijawab:
1. Metode mana yang paling baik dalam mempertahankan bentuk dan batas sel darah?
2. Jika tujuan segmentasi adalah menghitung jumlah sel darah, metode manakah yang paling efisien?

**📊 Tabel Perbandingan:**

| Metode | Kualitas Batas | Kemampuan Pisahkan Sel | Kompleksitas | Cocok untuk Counting? |
|--------|----------------|------------------------|--------------|------------------------|
| Canny  | ... | ... | ... | ... |
| Otsu   | ... | ... | ... | ... |
| SLIC   | ... | ... | ... | ... |

**📝 Kesimpulan:**
*(Tulis kesimpulan komprehensif Anda di sini)*

---

---

# BAGIAN 2: Deep Neural Network for Image Classification [40 poin]

## 🎯 Dataset: FashionMNIST

### 📚 Tentang Dataset:
- **Ukuran gambar**: 28×28 piksel (grayscale)
- **Jumlah kelas**: 10
- **Kelas**: T-shirt, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle Boot

### 🔧 Load Dataset:

In [ ]:
# ===== LOAD FASHIONMNIST =====

# Menggunakan PyTorch:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Normalize ke [-1, 1]
])

# Download dan load dataset
train_dataset = datasets.FashionMNIST(root='./data', train=True, 
                                      download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, 
                                     download=True, transform=transform)

# Split train menjadi train dan validation
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(
    train_dataset, [train_size, val_size]
)

# DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train size: {len(train_dataset)}")
print(f"Validation size: {len(val_dataset)}")
print(f"Test size: {len(test_dataset)}")

In [ ]:
# Visualisasi beberapa sampel
class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle Boot']

# TODO: Tampilkan beberapa sampel dari dataset
# ...

---

## 2(a) Implementasi CNN Baseline [10 poin]

### 🎯 Tujuan:
Membangun model CNN sederhana sebagai baseline

### 📚 Konsep Teori:

**Komponen dasar CNN:**
- **Conv2D**: Konvolusi untuk ekstraksi fitur
- **ReLU**: Activation function (non-linearity)
- **MaxPool2D**: Downsampling untuk mengurangi dimensi
- **Linear (Fully Connected)**: Classifier

### 🏗️ Arsitektur yang Disarankan:

In [ ]:
# ===== CONTOH ARSITEKTUR CNN BASELINE =====

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # TODO: Definisikan layer-layer CNN
        # Contoh struktur:
        # Conv2D → ReLU → MaxPool2D
        # Conv2D → ReLU → MaxPool2D
        # Flatten → Linear → ReLU → Linear (output)
        
        # Contoh layer pertama:
        # self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, 
        #                        kernel_size=3, padding=1)
        # self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # self.fc1 = nn.Linear(in_features=..., out_features=128)
        # self.fc2 = nn.Linear(in_features=128, out_features=10)
        
        pass
    
    def forward(self, x):
        # TODO: Definisikan forward pass
        # x = self.pool(torch.relu(self.conv1(x)))
        # ...
        pass

# Inisialisasi model
# model = SimpleCNN()
# print(model)

**📝 Jelaskan Arsitektur Anda:**
- Berapa jumlah layer konvolusi?
- Berapa kernel size yang digunakan?
- Berapa jumlah filter di setiap layer?
- Berapa total parameter model?

*(Tulis penjelasan di sini)*

In [ ]:
# ===== TRAINING LOOP =====

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

# TODO: Implementasi training loop
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    # ...
    
    # Validation phase
    # ...
    
    # Save history
    # ...
    
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: ... - Acc: ...")

In [ ]:
# TODO: Plot training history (akurasi dan loss)
# ...

**📝 Analisis Model Baseline:**
- Berapa akurasi tertinggi yang dicapai?
- Apakah model overfitting/underfitting?
- Bagaimana konvergensi training?

*(Tulis analisis di sini)*

---

## 2(b) Transfer Learning dan Variasi Model Pra-latih [20 poin]

### 🎯 Tujuan:
Menggunakan model pra-latih (pre-trained) dan membandingkan berbagai strategi fine-tuning

### 📚 Konsep Teori:

**Transfer Learning**:
- Menggunakan model yang sudah dilatih di ImageNet
- Memanfaatkan pengetahuan yang sudah dipelajari

**Strategi yang harus diuji (pilih 2 dari 4):**
- **(a) Freeze All Layers**: Hanya latih classifier (feature extraction)
- **(b) Freeze Sebagian**: Latih beberapa layer terakhir
- **(c) Train All**: Fine-tune seluruh model
- **(d) Random Init**: Gunakan arsitektur tapi tanpa pre-trained weights

### 🔧 Setup Pre-trained Model:

In [ ]:
# ===== LOAD PRE-TRAINED MODEL =====

# Contoh: ResNet18
from torchvision.models import resnet18, ResNet18_Weights

# Load dengan pre-trained weights
model_pretrained = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

# PENTING: Modifikasi layer pertama untuk grayscale (1 channel)
# ResNet18 default: 3 channels → 64 filters
# FashionMNIST: 1 channel (grayscale)
model_pretrained.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, 
                                   padding=3, bias=False)

# Modifikasi layer output untuk 10 kelas
# ResNet18 default: 1000 kelas (ImageNet)
num_features = model_pretrained.fc.in_features
model_pretrained.fc = nn.Linear(num_features, 10)

print(f"Model loaded with {num_features} input features to FC layer")
print(f"Output layer: {model_pretrained.fc}")

### 💡 Strategi (a): Freeze All Layers

In [ ]:
# ===== STRATEGI (a): FREEZE ALL =====

# TODO: Implementasi freeze all strategy

# Langkah:
# 1. Load model pre-trained
# 2. Freeze semua parameter
#    for param in model.parameters():
#        param.requires_grad = False
# 3. Pastikan hanya layer FC yang trainable
#    for param in model.fc.parameters():
#        param.requires_grad = True
# 4. Training dengan learning rate yang sesuai

# ...

### 💡 Strategi (pilih salah satu lagi dari b/c/d):

In [ ]:
# TODO: Implementasi strategi kedua yang Anda pilih
# ...

In [ ]:
# TODO: Bandingkan hasil dari kedua strategi
# Buat visualisasi:
# - Grafik akurasi train vs validation
# - Grafik loss train vs validation
# - Waktu training per epoch
# ...

**📝 Perbandingan Strategi Transfer Learning:**

| Metrik | Strategi (a) | Strategi (...) |
|--------|--------------|----------------|
| Akurasi Tertinggi | ... | ... |
| Waktu Training (per epoch) | ... | ... |
| Stabilitas (loss stability) | ... | ... |
| Convergence Speed | ... | ... |

**Kesimpulan:**
*(Tulis kesimpulan perbandingan Anda)*

---

## 2(c) Analisis dan Kesimpulan [10 poin]

### 🎯 Pertanyaan yang harus dijawab:
1. Model mana yang mencapai keseimbangan terbaik antara akurasi dan efisiensi?
2. Bagaimana karakteristik kesalahan model (kelas yang sering tertukar)?
3. Bagaimana perbandingan hasil antara model CNN sederhana dan model pra-latih?

### 📊 Confusion Matrix:

In [ ]:
# ===== CONFUSION MATRIX =====

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# TODO: Buat confusion matrix untuk model terbaik
# Langkah:
# 1. Predict pada test set
# 2. Hitung confusion matrix
# 3. Visualisasi dengan heatmap

# Contoh:
# y_pred = []
# y_true = []
# model.eval()
# with torch.no_grad():
#     for images, labels in test_loader:
#         outputs = model(images.to(device))
#         _, predicted = torch.max(outputs, 1)
#         y_pred.extend(predicted.cpu().numpy())
#         y_true.extend(labels.numpy())

# cm = confusion_matrix(y_true, y_pred)
# plt.figure(figsize=(10, 8))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
#             xticklabels=class_names, yticklabels=class_names)
# plt.xlabel('Predicted')
# plt.ylabel('True')
# plt.title('Confusion Matrix')
# plt.show()

# ...

**📝 Analisis Confusion Matrix:**
- Kelas mana yang paling sering tertukar?
- Mengapa kelas-kelas tersebut bisa tertukar?
- Apakah ada pola kesalahan yang konsisten?

*(Tulis analisis di sini)*

### 📊 Tabel Perbandingan Final:

| Model | Akurasi Test | Training Time (total) | Kelebihan | Kekurangan |
|-------|--------------|----------------------|-----------|------------|
| CNN Baseline | ... | ... | ... | ... |
| Transfer Learning (Strategi a) | ... | ... | ... | ... |
| Transfer Learning (Strategi ...) | ... | ... | ... | ... |

### 🏆 Rekomendasi Model Terbaik:

**Model yang direkomendasikan**: ...

**Alasan**:
1. ...
2. ...
3. ...

**Trade-off yang dipertimbangkan**:
- ...

---

## ✅ Selesai!

Pastikan Anda sudah:
- [ ] Mengerjakan semua soal dengan lengkap
- [ ] Menambahkan visualisasi untuk setiap tahap
- [ ] Memberikan penjelasan dan analisis yang jelas
- [ ] Menjawab semua pertanyaan yang diminta
- [ ] Membuat kesimpulan yang komprehensif